Business Executive Summary. This is designed to translate technical machine learning metrics into "Bank Language"—focusing on profit, efficiency, and customer targeting.

## Executive Summary: Term Deposit Subscription Model

After testing four different algorithms, XGBoost (Optimized) is our recommended model for deployment.

1. 🏆 Model Benchmark Comparison

| Model | Accuracy | Recall (Yes) | F1-Score (Yes) | Business Impact |
| :--- | :---: | :---: | :---: | :--- |
| **XGBoost** | **87.1%** | **69.0%** | **0.556** | **Best Balance:** Best overall balance of precision and catching subscribers. |
| Random Forest | 87.7% | 60.0% | 0.533 | "Conservative: High accuracy, but misses 40% of potential leads." |
| Logistic Regression | 80.7% | 83.7% | 0.504 | "Aggressive: Catches the most leads but with many ""false alarms.""" |
| KNN | 78.5% | 76.3% | 0.454 | Inconsistent: Low precision leads to high operational waste. |

XGBoost achieved the highest ability to find "Yes" customers while keeping "False Alarms" at a manageable level.

Why XGBoost? By using SMOTE to balance our data, we prioritized Recall. In banking, the profit from one successful term deposit significantly outweighs the cost of a few unsuccessful calls. 

2. Understanding the Prediction (Confusion Matrix)

<p align="center">
  <img src="../images/XGB_Confusion_Matrix.png" width="50%">
</p>


The Confusion Matrix tells us exactly how the model will perform in the real world:

True Positives (The Wins): The model correctly identifies customers ready to subscribe.

False Positives (The Costs): These are customers the model thought would say "yes," but said "no." This represents the call center's workload.

The Verdict: With our tuned model, the "Capture Rate" is high enough to justify the operational costs of the call campaign.

3. Key Drivers of Success (Feature Importance)

![Feature Importance](../images/Feature_importance.png)

Our analysis reveals that three main factors drive a customer to subscribe. This is where your Feature Engineering paid off:

Actionable Insights: XGBoost identifies contact_unknown and poutcome_success as primary drivers. This is huge for the bank because they can control who they call based on previous success, whereas they can't control duration until the call is already happening.

Feature Engineering Validation: Engineered feature, call_efficiency, is a top 2 driver in both models. This proves that data preparation was the "secret sauce" for this project's success.

Robustness: XGBoost spreads importance across more features (like specific months and job types), making it less likely to fail if one specific variable changes in the future.

Economic Timing (month & balance): Subscription rates spike during specific months and among customers with higher liquid balances.

4. Strategic Recommendations for the Bank

- Prioritize Leads: Use the XGBoost "Probability Scores" to call high-probability customers first thing in the morning when call center    energy is highest.

- Training: Since duration and call_efficiency are so important, the bank should invest in script training for agents to keep customers on the phone longer.

- Target the "Debt-Free": Our EDA showed that customers without housing or personal loans have a much higher conversion rate—focus marketing spend here.

**Final Step: Model Deployment**

To make this model usable by the bank's IT department, we need to save the "Brain" of the project.

Would you like the code to export your best_xgb model and the preprocessor as files so they can be used to predict new customers tomorrow?

## Technical Appendix: Model Tuning & Robustness

To achieve the final 69% Recall and 87.1% Accuracy, the XGBoost model was optimized using RandomizedSearchCV across a 5-fold cross-validation strategy. This process ensured that the model prioritized generalizable patterns over simply memorizing the training data.

1. Hyperparameter Logic

n_estimators=200: This provided an optimal number of boosting rounds to minimize residuals. Beyond this point, the model risked overfitting to the noise in the training set.

max_depth=5: By limiting the depth of each tree, we ensured the model captured critical interaction effects (e.g., how balance interacts with age) without creating a "Black Box" that is too complex to interpret.

learning_rate=0.1: This conservative shrinkage rate allowed the model to converge steadily toward the global optimum, preventing it from overshooting the best solution.

scale_pos_weight: This parameter was tuned to specifically penalize misclassifications of the minority "Yes" class, directly contributing to the superior F1-Score (0.556) compared to other models.

2. SMOTE (Synthetic Minority Over-sampling Technique)

The dataset exhibited a natural class imbalance (~88% "No"). We applied SMOTE to the training data to generate synthetic examples of the "Yes" class. Unlike standard oversampling, which simply duplicates rows, SMOTE creates new data points along the vectors between existing minority neighbors. This forced the model to learn the boundaries of what makes a subscriber, rather than just memorizing specific high-value leads.

3. SHAP (SHapley Additive exPlanations)

<p align="center">
  <img src="../images/SHAP_summary1.png" width="50%">
</p>

To ensure the model remains "Bank-Ready" and transparent, we utilized SHAP values. Based on cooperative game theory, SHAP calculates the marginal contribution of every feature for every single prediction.

Global Insight: It confirmed that call_efficiency and poutcome_success are the most powerful predictors.

Local Insight: It allowed us to see how specific attributes (like a high balance or a specific month) pushed a customer's probability score above the threshold for a "Yes" prediction.

## 🛠️ Production Logic: How the Model is Used

When this model is moved into production, it follows a strict sequence:

Preprocessing: New leads are passed through the saved preprocessor.joblib/.pkl to ensure the data format matches the training state.

Scoring: The model uses .predict_proba() to assign a Propensity Score (0.0 to 1.0) to each customer.

Prioritization: The bank sorts leads by these scores. Even with a 69% Recall, the bank can identify nearly 7 out of 10 subscribers by only calling a fraction of the total database, maximizing call center ROI.

In [ ]:
import joblib
import pandas as pd

# 1. Load the saved "Brain"
model = joblib.load('../models/joblib/xgb_subscription_model.joblib')
preprocessor = joblib.load('../models/joblib/preprocessor.joblib')

# 2. Load today's new leads (Real Data)
new_customers = pd.read_csv('todays_leads.csv') 

# 3. Transform data using the preprocessor
X_new = preprocessor.transform(new_customers)

# 4. Generate Propensity Scores
# Column 1 is the probability of "Yes" (Subscription)
probabilities = model.predict_proba(X_new)[:, 1]

# 5. Create the Final Call List
new_customers['Subscription_Probability'] = probabilities
call_list = new_customers.sort_values(by='Subscription_Probability', ascending=False)

# Export the top leads for the Sales Team
call_list.head(100).to_csv('top_100_leads_for_today.csv', index=False)
print("🚀 Today's Top 100 Leads have been generated!")

joblib is often faster for large NumPy arrays (like those in XGBoost), pickle is the standard Python built-in for object serialization.

In [ ]:
import pickle
import pandas as pd

# 1. Load the saved "Brain" (Now using Pickle)
# Note: 'rb' stands for 'read binary', which is required for pickle
with open('../models/pickle/xgb_subscription_model.pkl', 'rb') as f:
    model = pickle.load(f)

with open('../models/pickle/preprocessor.pkl', 'rb') as f:
    preprocessor = pickle.load(f)

# 2. Load today's new leads (Real Data)
# This file should contain the same columns as your training data (minus 'y')
new_customers = pd.read_csv('todays_leads.csv') 

# 3. Transform data using the preprocessor
# This handles the scaling and encoding for the 87.1% accuracy model
X_new = preprocessor.transform(new_customers)

# 4. Generate Propensity Scores
# We use [:, 1] to get the probability of 'Yes' (Subscription)
probabilities = model.predict_proba(X_new)[:, 1]

# 5. Create the Final Call List
new_customers['Subscription_Probability'] = probabilities

# Rank-order by probability to maximize the 69% Recall efficiency
call_list = new_customers.sort_values(by='Subscription_Probability', ascending=False)

# 6. Export the Top 100 Leads
# These are the customers most likely to convert based on your XGBoost 'Champion' model
call_list.head(100).to_csv('top_100_leads_for_today.csv', index=False)

print("🚀 Success! Today's Top 100 Leads have been generated based on XGBoost propensity scores.")

In [ ]:
# Export code uses pickle.dump:

import pickle

# Exporting the model
with open('../models/pickle/xgb_subscription_model.pkl', 'wb') as f:
    pickle.dump(best_xgb, f)

# Exporting the preprocessor
with open('../models/pickle/preprocessor.pkl', 'wb') as f:
    pickle.dump(preprocessor, f)